# Weekly Sector Rotation — feature engineering

Builds the feature panel below from the raw daily OHLCV data pulled in
`ETF_price_data.ipynb` (`data/raw/weekly_sector_rotation/prices/*.csv`).

| Feature group | Features |
| --- | --- |
| Absolute momentum | 1w, 2w, 4w, 8w, 12w, 26w returns |
| Relative momentum vs VTI | 1w, 2w, 4w, 8w, 12w, 26w excess returns vs VTI |
| Cross-sectional strength | rank of each sector's 1w, 4w, 8w, 12w return among the 11 sectors |
| Trend | distance from 4w, 8w, 12w, 26w moving average |
| Volatility | trailing 4w, 8w, 12w, 26w realized volatility |
| Drawdown | distance from trailing 13w and 26w high |
| Volume | 1w volume change, volume vs 4w/12w average |
| Market context | VTI 1w/4w/12w return, VTI volatility |
| Breadth / rotation | dispersion of returns across the 11 sectors, average sector return, number of sectors outperforming VTI |

`abs_return_26w`/`excess_return_vs_vti_26w`, `realized_vol_26w`, and
`average_sector_return_1w` were added after the initial pass to match the
feature-group taxonomy (A-I) used for the diagnostics/ablation research
that follows in `feature_diagnostics.ipynb`; `8w` windows remain in the
saved panel even where that taxonomy's groups don't reference them,
rather than being deleted, since they were already validated features in
their own right.

**Design decisions (confirmed with user):**
- Daily data is resampled to **weekly bars** first (`W-FRI`: last trading
  day observed in each week); every "Nw" window below is N *weekly* bars,
  not N*5 trading days.
- **`adjusted_close`** feeds every return, moving-average, drawdown, and
  volatility calculation — for momentum consistency, so a dividend/split
  never masquerades as a trend break or a vol/drawdown spike.
- The **current, not-yet-finished week is dropped**. A `W-FRI` resample
  labels the in-progress week with its (future) Friday even though the
  week hasn't finished trading yet; keeping that bar would treat a partial
  week's close as a final one and corrupt every feature computed off it.
- **Price and volume are aggregated differently** when resampling daily ->
  weekly: `adjusted_close` is the *last* trading day's value that week
  (a closing level), `volume` is the *sum* of daily volume across the week
  (aggregate weekly participation) — using Friday's volume alone for the
  weekly volume observation would badly understate a week's real turnover.
- **`volume_vs_avg_{4,12}w`'s denominator excludes the current week**
  (`shift(1)` before the rolling mean, i.e. it averages weeks `t-1..t-n`,
  not `t..t-n+1`) so the feature reads as "how unusual is this week's
  volume relative to what was already known before this week," not
  relative to a window that already contains this week's own value.
- **Trend** (`trend_dist_from_ma_Nw`) is a ratio, `P_t / MA_N - 1`, not a
  raw dollar distance `P_t - MA_N` — comparable across ETFs at different
  price levels; `+0.043` means 4.3% above the N-week MA.
- **Realized volatility** (`realized_vol_Nw`, and `vti_volatility_4w`) is
  the trailing standard deviation of **weekly log returns**,
  `r_t = ln(P_t / P_{t-1})` — not simple returns. Stored **unannualized**;
  the `*sqrt(52)` annualizing scale factor is a constant multiplier that
  adds no information for an ML feature, so it's deliberately omitted.
  This is a permanent convention, not a per-notebook choice.

**Additional conventions (documented here, not silently assumed):**
- Weeks are kept only where **all 12 symbols** have data, so cross-sectional
  features (rank, dispersion, breadth) are always well-defined. This trims
  the panel's start to late-2004, the common inception of the four
  youngest sector ETFs (VOX/VIS/VNQ/VDE).
- Cross-sectional rank: **1 = strongest** (highest N-week return) among the
  11 sectors that date.
- "Sectors outperforming VTI" (breadth) is evaluated on the 1w return —
  the spec doesn't pin a window for it.
- All rolling/shift windows use only current-and-past weekly bars — no
  centering, no future data.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parents[3]
PRICES_ROOT = REPO_ROOT / "data" / "raw" / "weekly_sector_rotation" / "prices"

SECTOR_ETFS = [
    "VGT", "VHT", "VCR", "VOX", "VFH", "VIS", "VDC", "VPU", "VAW", "VNQ", "VDE",
]
BENCHMARK_ETF = "VTI"
SYMBOLS = SECTOR_ETFS + [BENCHMARK_ETF]

daily = {}
for symbol in SYMBOLS:
    df = pd.read_csv(PRICES_ROOT / f"{symbol}.csv", parse_dates=["date"])
    daily[symbol] = df.set_index("date").sort_index()

daily[BENCHMARK_ETF].tail()

,symbol,open,high,low,close,adjusted_close,volume
date,,,,,,,
2026-08-06,VTI,379.820007,380.940002,378.640015,379.070007,379.070007,3090000
2026-08-07,VTI,380.690002,382.070007,379.929993,381.779999,381.779999,2317900
2026-08-10,VTI,381.630005,382.660004,381.100006,381.630005,381.630005,5170900
2026-08-11,VTI,382.510010,382.600006,380.190002,380.649994,380.649994,3900100
2026-08-12,VTI,382.750000,382.820007,381.149994,381.829987,381.829987,2615431


In [2]:
def resample_weekly(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse one symbol's daily frame to one bar per week (``W-FRI``).

    ``adjusted_close`` is the last daily value observed in the week (the
    week's closing level); ``volume`` is the sum of daily volume across
    the week.
    """
    weekly = pd.DataFrame(
        {
            "adjusted_close": df["adjusted_close"].resample("W-FRI").last(),
            "volume": df["volume"].resample("W-FRI").sum(),
        }
    )
    return weekly.dropna(subset=["adjusted_close"])


weekly = {symbol: resample_weekly(df) for symbol, df in daily.items()}

# Drop the in-progress week's bar. A `W-FRI` resample labels the current, still-open week
# with its (future) Friday even though the week hasn't finished trading yet -- keeping that
# bar would treat a partial week's close as if it were final.
last_trading_date = max(df.index.max() for df in daily.values())
current_week_end = last_trading_date + pd.Timedelta(days=(4 - last_trading_date.weekday()) % 7)
if last_trading_date < current_week_end:
    print(f"Dropping unfinished week ending {current_week_end.date()} -- data only through {last_trading_date.date()}")
    for w in weekly.values():
        w.drop(index=current_week_end, errors="ignore", inplace=True)

adj_close = pd.DataFrame({symbol: w["adjusted_close"] for symbol, w in weekly.items()}).sort_index()
volume = pd.DataFrame({symbol: w["volume"] for symbol, w in weekly.items()}).sort_index()

# Only keep weeks where every symbol has data -- cross-sectional features need all 11 sectors + VTI present.
adj_close = adj_close.dropna(how="any")
volume = volume.loc[adj_close.index]

sector_adj_close = adj_close[SECTOR_ETFS]
vti_adj_close = adj_close[BENCHMARK_ETF]
sector_volume = volume[SECTOR_ETFS]

print(f"{len(adj_close)} common weekly bars, {adj_close.index.min().date()} -> {adj_close.index.max().date()}")
adj_close.tail()

Dropping unfinished week ending 2026-08-14 -- data only through 2026-08-12
1141 common weekly bars, 2004-10-01 -> 2026-08-07


,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE,VTI
date,,,,,,,,,,,,
2026-07-10,118.080002,303.459991,395.750000,191.520004,136.449997,350.380005,228.139999,196.169998,226.740005,97.320000,155.500000,372.690002
2026-07-17,113.099998,303.250000,391.549988,188.080002,137.940002,345.299988,230.320007,195.220001,223.610001,100.019997,162.779999,367.010010
2026-07-24,113.300003,305.070007,372.010010,179.809998,137.809998,350.170013,227.320007,199.460007,226.789993,100.809998,168.229996,364.799988
2026-07-31,113.150002,304.809998,391.299988,181.779999,139.229996,344.440002,230.429993,191.220001,222.869995,98.949997,166.279999,368.209991
2026-08-07,121.449997,312.119995,403.040009,187.520004,140.899994,356.829987,231.220001,188.860001,236.410004,98.430000,162.369995,381.779999


## Absolute momentum

In [3]:
MOMENTUM_WINDOWS = [1, 2, 4, 8, 12, 26]


def n_week_return(price: pd.DataFrame | pd.Series, n: int):
    """(price_t / price_{t-n}) - 1, using only bars up to and including t."""
    return price / price.shift(n) - 1


abs_momentum = {n: n_week_return(sector_adj_close, n) for n in MOMENTUM_WINDOWS}
vti_return = {n: n_week_return(vti_adj_close, n) for n in MOMENTUM_WINDOWS}

abs_momentum[4].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,0.012683,0.063005,0.009447,0.024774,0.048575,0.020122,-0.012411,0.023790,-0.032303,-0.003373,-0.035700
2026-07-17,-0.056693,0.085188,-0.005217,0.008953,0.056235,-0.021227,0.024100,0.013264,-0.042036,0.055898,0.079392
2026-07-24,-0.004918,0.012076,-0.042692,-0.005476,0.048304,-0.006525,-0.010146,-0.001002,-0.023004,0.021688,0.106267
2026-07-31,-0.012997,-0.011673,-0.010745,-0.032313,0.022397,-0.030102,-0.000087,-0.031993,-0.041996,0.009488,0.105291
2026-08-07,0.028540,0.028538,0.018421,-0.020886,0.032613,0.018409,0.013500,-0.037264,0.042648,0.011406,0.044180


## Relative momentum vs VTI

In [4]:
# Excess return = sector's N-week return minus VTI's N-week return, same window.
rel_momentum = {n: abs_momentum[n].sub(vti_return[n], axis=0) for n in MOMENTUM_WINDOWS}

rel_momentum[4].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,-0.007522,0.042800,-0.010757,0.004570,0.028371,-0.000083,-0.032615,0.003586,-0.052508,-0.023577,-0.055904
2026-07-17,-0.051492,0.090389,-0.000016,0.014154,0.061436,-0.016026,0.029301,0.018465,-0.036835,0.061098,0.084593
2026-07-24,-0.012041,0.004953,-0.049814,-0.012598,0.041181,-0.013648,-0.017269,-0.008124,-0.030127,0.014566,0.099144
2026-07-31,-0.011506,-0.010181,-0.009253,-0.030822,0.023888,-0.028610,0.001405,-0.030502,-0.040505,0.010979,0.106783
2026-08-07,0.004150,0.004147,-0.005969,-0.045276,0.008222,-0.005982,-0.010890,-0.061654,0.018258,-0.012985,0.019790


## Cross-sectional strength (rank among the 11 sectors, per date; 1 = strongest)

In [5]:
RANK_WINDOWS = [1, 4, 8, 12]

cross_sectional_rank = {
    n: abs_momentum[n].rank(axis=1, ascending=False, method="average") for n in RANK_WINDOWS
}

cross_sectional_rank[4].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,6.0,1.0,7.0,3.0,2.0,5.0,9.0,4.0,10.0,8.0,11.0
2026-07-17,11.0,1.0,8.0,7.0,3.0,9.0,5.0,6.0,10.0,4.0,2.0
2026-07-24,6.0,4.0,11.0,7.0,2.0,8.0,9.0,5.0,10.0,3.0,1.0
2026-07-31,7.0,6.0,5.0,10.0,2.0,8.0,4.0,9.0,11.0,3.0,1.0
2026-08-07,4.0,5.0,6.0,10.0,3.0,7.0,8.0,11.0,2.0,9.0,1.0


## Trend (distance from moving average)

In [6]:
TREND_WINDOWS = [4, 8, 12, 26]


def distance_from_ma(price: pd.DataFrame, n: int) -> pd.DataFrame:
    """(price_t / trailing_n_week_MA_t) - 1, MA computed over bars up to and including t."""
    return price / price.rolling(n).mean() - 1


trend = {n: distance_from_ma(sector_adj_close, n) for n in TREND_WINDOWS}

trend[12].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,0.035038,0.072485,0.007739,0.000834,0.058481,0.022328,0.000279,0.008094,-0.017696,0.014040,-0.021967
2026-07-17,-0.015135,0.061027,-0.002453,-0.014404,0.061719,0.005057,0.008921,0.004630,-0.027173,0.037172,0.022675
2026-07-24,-0.019494,0.056706,-0.047882,-0.050929,0.053307,0.015933,-0.003114,0.026594,-0.011816,0.040340,0.055904
2026-07-31,-0.021388,0.044721,0.002861,-0.033969,0.054815,-0.002213,0.009999,-0.015227,-0.025512,0.018365,0.038798
2026-08-07,0.044170,0.056966,0.028761,-0.000068,0.056518,0.027951,0.013015,-0.027348,0.029841,0.008395,0.016959


## Volatility (trailing realized volatility of weekly returns, not annualized)

In [7]:
VOL_WINDOWS = [4, 8, 12, 26]

# Realized volatility uses log returns, r_t = ln(P_t / P_{t-1}), not simple returns -- log
# returns are time-additive, which is the standard basis for a realized-volatility estimator.
# Stored unannualized (trailing StdDev of weekly log returns): the *52 annualizing scale
# factor is a constant multiplier that carries no additional information for an ML feature,
# so it's deliberately omitted here -- this choice is permanent, not left ambiguous per-notebook.
sector_weekly_log_return = np.log(sector_adj_close / sector_adj_close.shift(1))
vti_weekly_log_return = np.log(vti_adj_close / vti_adj_close.shift(1))

volatility = {n: sector_weekly_log_return.rolling(n).std() for n in VOL_WINDOWS}

volatility[8].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,0.036174,0.030595,0.024073,0.026189,0.012964,0.012048,0.020274,0.019761,0.019144,0.023602,0.034839
2026-07-17,0.039147,0.030195,0.022719,0.026747,0.012806,0.014118,0.020431,0.016395,0.019704,0.023228,0.039949
2026-07-24,0.034330,0.030027,0.027339,0.029616,0.012040,0.014174,0.018359,0.015794,0.018721,0.022270,0.035597
2026-07-31,0.030117,0.030241,0.029976,0.027104,0.012046,0.015848,0.018660,0.023106,0.018332,0.024003,0.035789
2026-08-07,0.039699,0.030660,0.030618,0.029791,0.011213,0.019654,0.015707,0.023475,0.026390,0.023488,0.037115


## Drawdown (distance from trailing high)

In [8]:
DRAWDOWN_WINDOWS = [13, 26]


def distance_from_high(price: pd.DataFrame, n: int) -> pd.DataFrame:
    """(price_t / trailing_n_week_high_t) - 1; <= 0, trailing high includes t itself."""
    return price / price.rolling(n).max() - 1


drawdown = {n: distance_from_high(sector_adj_close, n) for n in DRAWDOWN_WINDOWS}

drawdown[26].tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,-0.023455,-0.016050,-0.023362,-0.035461,0.000000,-0.013375,-0.055533,-0.034429,-0.064536,-0.013682,-0.115318
2026-07-17,-0.064640,-0.016731,-0.033727,-0.052786,0.000000,-0.027680,-0.046508,-0.039105,-0.077449,0.000000,-0.073900
2026-07-24,-0.062986,-0.010830,-0.072201,-0.094435,-0.000942,-0.013967,-0.058928,-0.018235,-0.064330,0.000000,-0.042894
2026-07-31,-0.064227,-0.011673,-0.024092,-0.084514,0.000000,-0.030102,-0.046053,-0.058793,-0.080503,-0.018451,-0.053988
2026-08-07,0.000000,0.000000,0.000000,-0.055606,0.000000,0.000000,-0.042782,-0.070410,-0.024640,-0.023609,-0.076233


## Volume

In [9]:
# volume_change_1w: this week's volume vs last week's, (Volume_t / Volume_{t-1}) - 1.
volume_change_1w = sector_volume.pct_change(1)


def volume_vs_prior_avg(volume: pd.DataFrame, n: int) -> pd.DataFrame:
    """(Volume_t / mean(Volume_{t-1}, ..., Volume_{t-n})) - 1.

    The average deliberately excludes week t itself (``shift(1)`` before the
    rolling mean) so the ratio reads as "how unusual is this week's volume
    relative to the n weeks *before* it" rather than relative to a window
    that already contains this week's own value.
    """
    prior_avg = volume.shift(1).rolling(n).mean()
    return volume / prior_avg - 1


volume_vs_avg = {n: volume_vs_prior_avg(sector_volume, n) for n in [4, 12]}

volume_change_1w.tail()

,VGT,VHT,VCR,VOX,VFH,VIS,VDC,VPU,VAW,VNQ,VDE
date,,,,,,,,,,,
2026-07-10,0.350669,0.044275,1.500257,-0.070845,-0.024514,-0.482911,-0.289397,-0.066377,-0.359249,0.067122,-0.588713
2026-07-17,0.240798,-0.116938,-0.579683,-0.526891,0.402644,-0.311951,-0.420613,-0.013672,0.225418,-0.217300,-0.084331
2026-07-24,-0.448842,-0.402585,0.778865,0.868928,-0.476838,0.141794,0.408247,0.084608,-0.160905,-0.151687,-0.347482
2026-07-31,0.550710,0.595701,-0.236524,0.165086,-0.128457,0.061782,-0.062824,0.097889,0.107833,0.342946,0.320742
2026-08-07,-0.153408,-0.153239,0.008285,-0.451161,0.016729,0.704678,0.051839,0.729240,0.103765,0.025175,0.110325


## Market context and breadth / rotation (date-level, broadcast to every sector row)

In [10]:
market_context = pd.DataFrame(
    {
        "vti_return_1w": vti_return[1],
        "vti_return_4w": vti_return[4],
        "vti_return_12w": vti_return[12],
        "vti_volatility_4w": vti_weekly_log_return.rolling(4).std(),
    }
)

# gt()/sum() silently treats a NaN vs. NaN comparison as False rather than propagating NaN,
# which would otherwise manufacture a defined-looking 0 out of genuinely undefined inputs on
# the very first week (no prior week exists yet) -- explicitly re-mask to NaN wherever the
# underlying 1w return itself isn't defined yet, instead of relying on gt()'s silent behavior.
sectors_outperforming_vti_1w = abs_momentum[1].gt(vti_return[1], axis=0).sum(axis=1).where(vti_return[1].notna())

breadth = pd.DataFrame(
    {
        "sector_return_dispersion_1w": abs_momentum[1].std(axis=1),
        "sectors_outperforming_vti_1w": sectors_outperforming_vti_1w,
        "average_sector_return_1w": abs_momentum[1].mean(axis=1),
    }
)

market_context.join(breadth).tail()

,vti_return_1w,vti_return_4w,vti_return_12w,vti_volatility_4w,sector_return_dispersion_1w,sectors_outperforming_vti_1w,average_sector_return_1w
date,,,,,,,
2026-07-10,0.010657,0.020204,0.066277,0.015985,0.019216,3.0,0.000616
2026-07-17,-0.015241,-0.005201,0.045493,0.018243,0.024088,9.0,-0.000868
2026-07-24,-0.006022,0.007123,0.029720,0.015191,0.025851,8.0,-0.000786
2026-07-31,0.009348,-0.001492,0.017635,0.012526,0.024114,4.0,-0.001853
2026-08-07,0.036854,0.024390,0.055517,0.022535,0.029866,2.0,0.020905


## Audit trail (not model features)

These columns exist to prove timing correctness, not to feed a model:

- `week_end` — the weekly bar's own label (same value as `date`, named
  explicitly so a reader doesn't have to assume `date` means "Friday of
  this week").
- `actual_last_trading_date` — the real last trading day observed inside
  that week (usually Friday, earlier on a holiday-shortened week).
- `week_complete` — `True` iff a later trading day exists beyond this
  week (proving it's over) or `actual_last_trading_date` lands exactly on
  `week_end`. Every row in the saved panel should already be `True`, since
  the in-progress week was dropped earlier — this column makes that
  invariant checkable rather than assumed.
- `feature_cutoff_timestamp` — `actual_last_trading_date` at the NYSE
  close (16:00, naive/no timezone, matching the rest of this dataset) —
  the exact point after which this row's features must not use any
  information.
- `next_week_first_trading_date` / `next_week_last_trading_date` — the
  following week's trading bounds, used only to state the "no lookahead"
  rule below; `next_week_last_trading_date` is left `NaT` whenever that
  next week is itself not yet complete (this is exactly the situation for
  the most recent surviving row today: the next week has started —
  `next_week_first_trading_date` is known — but hasn't finished, so its
  last trading date genuinely isn't knowable yet and must not be
  fabricated).

The trading calendar used to derive these is the union of every symbol's
own daily dates (not just VTI's), so a single symbol's data gap can't
silently distort the audit columns.

In [11]:
all_trading_dates = pd.DatetimeIndex(sorted(set().union(*(df.index for df in daily.values()))))
trading_calendar = pd.Series(all_trading_dates, index=all_trading_dates)

week_bounds = trading_calendar.resample("W-FRI").agg(["first", "last"])
week_bounds.columns = ["week_first_trading_date", "week_last_trading_date"]

# A week is complete if a later trading day exists beyond it (proving it's over), or its own
# last trading day lands exactly on the week-end label (Friday, or the calendar's final day).
last_bucket_label = week_bounds.index.max()
week_bounds["week_complete"] = (week_bounds.index != last_bucket_label) | (
    week_bounds["week_last_trading_date"] == last_bucket_label
)

next_week_first = week_bounds["week_first_trading_date"].shift(-1)
next_week_last = week_bounds["week_last_trading_date"].shift(-1).where(week_bounds["week_complete"].shift(-1))

audit = pd.DataFrame(
    {
        "week_end": adj_close.index,
        "actual_last_trading_date": week_bounds.loc[adj_close.index, "week_last_trading_date"],
        "week_complete": week_bounds.loc[adj_close.index, "week_complete"],
        "feature_cutoff_timestamp": week_bounds.loc[adj_close.index, "week_last_trading_date"]
        + pd.Timedelta(hours=16),
        "next_week_first_trading_date": next_week_first.loc[adj_close.index],
        "next_week_last_trading_date": next_week_last.loc[adj_close.index],
    },
    index=adj_close.index,
)
audit.index.name = "date"

audit.tail()

,week_end,actual_last_trading_date,week_complete,feature_cutoff_timestamp,next_week_first_trading_date,next_week_last_trading_date
date,,,,,,
2026-07-10,2026-07-10,2026-07-10,True,2026-07-10 16:00:00,2026-07-13,2026-07-17
2026-07-17,2026-07-17,2026-07-17,True,2026-07-17 16:00:00,2026-07-20,2026-07-24
2026-07-24,2026-07-24,2026-07-24,True,2026-07-24 16:00:00,2026-07-27,2026-07-31
2026-07-31,2026-07-31,2026-07-31,True,2026-07-31 16:00:00,2026-08-03,2026-08-07
2026-08-07,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT


## Combine into one feature panel and save

In [12]:
def stack_sector_frame(frame: pd.DataFrame, name: str) -> pd.Series:
    series = frame.stack()
    series.index.names = ["date", "symbol"]
    series.name = name
    return series


sector_feature_series = []
for n in MOMENTUM_WINDOWS:
    sector_feature_series.append(stack_sector_frame(abs_momentum[n], f"abs_return_{n}w"))
for n in MOMENTUM_WINDOWS:
    sector_feature_series.append(stack_sector_frame(rel_momentum[n], f"excess_return_vs_vti_{n}w"))
for n in RANK_WINDOWS:
    sector_feature_series.append(stack_sector_frame(cross_sectional_rank[n], f"cross_sectional_rank_{n}w"))
for n in TREND_WINDOWS:
    sector_feature_series.append(stack_sector_frame(trend[n], f"trend_dist_from_ma_{n}w"))
for n in VOL_WINDOWS:
    sector_feature_series.append(stack_sector_frame(volatility[n], f"realized_vol_{n}w"))
for n in DRAWDOWN_WINDOWS:
    sector_feature_series.append(stack_sector_frame(drawdown[n], f"drawdown_dist_from_high_{n}w"))
sector_feature_series.append(stack_sector_frame(volume_change_1w, "volume_change_1w"))
for n in (4, 12):
    sector_feature_series.append(stack_sector_frame(volume_vs_avg[n], f"volume_vs_avg_{n}w"))

feature_panel = pd.concat(sector_feature_series, axis=1).reset_index()

# Market-context / breadth / audit columns are date-level, not sector-specific -- broadcast onto every sector's row for that date.
feature_panel = feature_panel.merge(market_context.reset_index(), on="date", how="left")
feature_panel = feature_panel.merge(breadth.reset_index(), on="date", how="left")
feature_panel = feature_panel.merge(audit.reset_index(), on="date", how="left")

feature_panel = feature_panel.sort_values(["date", "symbol"]).reset_index(drop=True)

AUDIT_COLUMNS = [
    "week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
    "next_week_first_trading_date", "next_week_last_trading_date",
]

FEATURES_ROOT = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation"
FEATURES_ROOT.mkdir(parents=True, exist_ok=True)
out_path = FEATURES_ROOT / "feature_panel.csv"
feature_panel.to_csv(out_path, index=False)

print(f"{feature_panel.shape[0]:,} rows x {feature_panel.shape[1]} columns -> {out_path}")
print(f"audit columns (not model features): {AUDIT_COLUMNS}")
feature_panel.tail(15)

12,551 rows x 44 columns -> /Users/gavinhussey/Downloads/AtlasQuant/data/processed/weekly_sector_rotation/feature_panel.csv
audit columns (not model features): ['week_end', 'actual_last_trading_date', 'week_complete', 'feature_cutoff_timestamp', 'next_week_first_trading_date', 'next_week_last_trading_date']


,date,symbol,abs_return_1w,abs_return_2w,abs_return_4w,abs_return_8w,abs_return_12w,abs_return_26w,excess_return_vs_vti_1w,excess_return_vs_vti_2w,...,vti_volatility_4w,sector_return_dispersion_1w,sectors_outperforming_vti_1w,average_sector_return_1w,week_end,actual_last_trading_date,week_complete,feature_cutoff_timestamp,next_week_first_trading_date,next_week_last_trading_date
12536,2026-07-31,VIS,-0.016364,-0.002491,-0.030102,0.015155,0.018660,0.080687,-0.025711,-0.005760,...,0.012526,0.024114,4.0,-0.001853,2026-07-31,2026-07-31,True,2026-07-31 16:00:00,2026-08-03,2026-08-07
12537,2026-07-31,VNQ,-0.018451,-0.010698,0.009488,0.031327,0.033142,0.111115,-0.027798,-0.013967,...,0.012526,0.024114,4.0,-0.001853,2026-07-31,2026-07-31,True,2026-07-31 16:00:00,2026-08-03,2026-08-07
12538,2026-07-31,VOX,0.010956,-0.033496,-0.032313,-0.033430,-0.078309,-0.084230,0.001608,-0.036766,...,0.012526,0.024114,4.0,-0.001853,2026-07-31,2026-07-31,True,2026-07-31 16:00:00,2026-08-03,2026-08-07
12539,2026-07-31,VPU,-0.041312,-0.020490,-0.031993,0.000706,-0.007242,0.030726,-0.050659,-0.023759,...,0.012526,0.024114,4.0,-0.001853,2026-07-31,2026-07-31,True,2026-07-31 16:00:00,2026-08-03,2026-08-07
12540,2026-08-07,VAW,0.060753,0.042418,0.042648,0.008967,0.045380,0.014592,0.023899,-0.004128,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT
12541,2026-08-07,VCR,0.030003,0.083412,0.018421,0.028042,0.049635,0.036414,-0.006851,0.036866,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT
12542,2026-08-07,VDC,0.003428,0.017156,0.013500,0.000922,0.005250,-0.023304,-0.033426,-0.029390,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT
12543,2026-08-07,VDE,-0.023515,-0.034833,0.044180,0.006903,-0.029216,0.096810,-0.060369,-0.081379,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT
12544,2026-08-07,VFH,0.011995,0.022422,0.032613,0.082772,0.131859,0.069360,-0.024859,-0.024124,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT
12545,2026-08-07,VGT,0.073354,0.071933,0.028540,0.041584,0.073114,0.316721,0.036500,0.025387,...,0.022535,0.029866,2.0,0.020905,2026-08-07,2026-08-07,True,2026-08-07 16:00:00,2026-08-10,NaT


## Sanity checks

`sectors_outperforming_vti_1w` is $\sum_{s=1}^{11} \mathbb{1}(R_{s,t} > R_{VTI,t})$
over the 11 sectors, so it must fall in $[0, 11]$ for every date, and it must
equal a direct count of `excess_return_vs_vti_1w > 0` for that same date
(they're two different code paths computing the same quantity — one from
`abs_momentum[1] > vti_return[1]`, the other from the already-subtracted
`excess_return_vs_vti_1w` column — so agreement here also catches a broadcast
or join bug, not just an out-of-range value).

In [13]:
# dropna() first: the very first week has no prior week at all, so sectors_outperforming_vti_1w
# is legitimately NaN there, not an out-of-range value (NaN.between(...) is False, not NaN).
assert feature_panel["sectors_outperforming_vti_1w"].dropna().between(0, 11).all(), "breadth count out of [0, 11] range"

# Independently recompute breadth from excess_return_vs_vti_1w (a different column, joined via
# a separate merge) and require it to match the reported count for every date. Restricted to
# dates where the reported column is defined, for the same first-week reason as above.
recomputed_breadth = (
    feature_panel.assign(_beats_vti=feature_panel["excess_return_vs_vti_1w"] > 0)
    .groupby("date")["_beats_vti"]
    .sum()
)
reported_breadth = feature_panel.groupby("date")["sectors_outperforming_vti_1w"].first()
comparable = reported_breadth.notna()
mismatches = (recomputed_breadth[comparable] != reported_breadth[comparable]).sum()
assert mismatches == 0, f"{mismatches} dates where recomputed breadth disagrees with the reported column"

last_date = feature_panel["date"].max()
last_week = feature_panel[feature_panel["date"] == last_date]
print(f"range check passed; recomputed-vs-reported breadth agrees for all {comparable.sum()} defined dates "
      f"({(~comparable).sum()} first-week date(s) correctly left undefined)")
print(f"last complete week: {last_date}")
last_week[["symbol", "abs_return_1w", "excess_return_vs_vti_1w", "sectors_outperforming_vti_1w"]].sort_values(
    "excess_return_vs_vti_1w", ascending=False
)

range check passed; recomputed-vs-reported breadth agrees for all 1140 defined dates (1 first-week date(s) correctly left undefined)
last complete week: 2026-08-07 00:00:00


,symbol,abs_return_1w,excess_return_vs_vti_1w,sectors_outperforming_vti_1w
12545,VGT,0.073354,0.036500,2.0
12540,VAW,0.060753,0.023899,2.0
12547,VIS,0.035971,-0.000883,2.0
12549,VOX,0.031577,-0.005277,2.0
12541,VCR,0.030003,-0.006851,2.0
12546,VHT,0.023982,-0.012872,2.0
12544,VFH,0.011995,-0.024859,2.0
12542,VDC,0.003428,-0.033426,2.0
12548,VNQ,-0.005255,-0.042109,2.0
12550,VPU,-0.012342,-0.049196,2.0


In [14]:
# Drawdown = price / trailing rolling high - 1: 0 at the rolling high, negative below it, never positive.
# NaN rows are the expected warm-up period before a window has 13/26 weeks of history yet, not violations.
for window, col in [(13, "drawdown_dist_from_high_13w"), (26, "drawdown_dist_from_high_26w")]:
    values = feature_panel[col].dropna()
    assert (values <= 1e-10).all(), f"{col} has a positive value -- price above its own trailing high"

print("drawdown sanity check passed: both columns are <= 0 everywhere they're defined")

drawdown sanity check passed: both columns are <= 0 everywhere they're defined


In [15]:
# Audit-trail invariants: every surviving week must be provably complete, and its feature
# cutoff must always precede the start of the period a decision made from it would act on.
assert feature_panel["week_complete"].all(), "a row claims to be a finished week but isn't"

with_target = feature_panel.dropna(subset=["next_week_first_trading_date"])
assert (with_target["feature_cutoff_timestamp"] < with_target["next_week_first_trading_date"]).all(), (
    "FeatureCutoff < TargetPeriodStart violated -- a row's features could see data from its own target period"
)

print(f"timing rule holds for all {len(feature_panel)} rows "
      f"({feature_panel['next_week_first_trading_date'].isna().sum()} rows have an unknown/future target period)")

timing rule holds for all 12551 rows (0 rows have an unknown/future target period)


## Structural and cross-consistency integrity checks

Hard assertions on the saved panel, run before any statistical testing.

In [16]:
EXPECTED_SYMBOLS = set(SECTOR_ETFS)
FEATURE_COLUMNS = [c for c in feature_panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]
WARMUP_WEEKS = 26  # longest window used anywhere (26w momentum/vol/trend/drawdown)

# 1. Exactly 11 sector rows per completed week, no duplicate (date, symbol), symbol set is exact.
rows_per_week = feature_panel.groupby("date").size()
assert (rows_per_week == 11).all(), f"weeks with != 11 rows: {rows_per_week[rows_per_week != 11].to_dict()}"

dup_count = feature_panel.duplicated(subset=["date", "symbol"]).sum()
assert dup_count == 0, f"{dup_count} duplicate (date, symbol) rows"

assert set(feature_panel["symbol"].unique()) == EXPECTED_SYMBOLS, "unexpected/missing symbol in panel"

# 2. No future/incomplete weeks -- every row's own week must be provably finished, and its
#    cutoff must sit at or before the trading calendar's own last known date.
assert feature_panel["week_complete"].all(), "an incomplete week survived into the panel"
assert (
    pd.to_datetime(feature_panel["feature_cutoff_timestamp"]).dt.date <= all_trading_dates.max().date()
).all(), "a row's feature cutoff is in the future relative to the pulled trading calendar"

# 3. Range checks already exercised earlier in this notebook (kept here too, as a single
#    consolidated integrity gate rather than relying on the earlier ad hoc cells). dropna()
#    first: sectors_outperforming_vti_1w and the drawdown columns are legitimately NaN during
#    warmup (e.g. the very first week has no prior week to compare against at all), and
#    NaN.between(...) evaluates to False rather than NaN, which would otherwise fail this
#    assertion for a warmup row that was never a real out-of-range value.
assert feature_panel["sectors_outperforming_vti_1w"].dropna().between(0, 11).all()
assert (feature_panel["drawdown_dist_from_high_13w"].dropna() <= 1e-10).all()
assert (feature_panel["drawdown_dist_from_high_26w"].dropna() <= 1e-10).all()

# 4. Cross-sectional rank validity: each date's set of 11 ranks must sum to 1+2+...+11 = 66
#    (holds even with tied ranks under method="average", since averaging preserves the sum).
for n in RANK_WINDOWS:
    col = f"cross_sectional_rank_{n}w"
    rank_sums = feature_panel.dropna(subset=[col]).groupby("date")[col].sum()
    bad = rank_sums[~np.isclose(rank_sums, 66.0)]
    assert bad.empty, f"{col}: {len(bad)} dates with rank sum != 66"
    values = feature_panel[col].dropna()
    assert values.between(1, 11).all(), f"{col}: rank outside [1, 11]"

# 5. VTI context and breadth are date-level and must be identical across all 11 rows for that date.
for col in ["vti_return_1w", "vti_return_4w", "vti_return_12w", "vti_volatility_4w",
            "sector_return_dispersion_1w", "sectors_outperforming_vti_1w", "average_sector_return_1w"]:
    nunique_per_date = feature_panel.groupby("date")[col].nunique(dropna=False)
    assert (nunique_per_date <= 1).all(), f"{col} varies across sectors within a single date"

# 6. excess_return_vs_vti_Nw must equal abs_return_Nw - vti_return_Nw exactly (same source data,
#    two different columns -- this also guards against a bad merge/broadcast).
for n in MOMENTUM_WINDOWS:
    vti_col = f"vti_return_{n}w" if n in (1, 4, 12) else None
    if vti_col is None:
        continue  # only 1w/4w/12w VTI return columns are carried in the saved panel
    recomputed = feature_panel[f"abs_return_{n}w"] - feature_panel[vti_col]
    reported = feature_panel[f"excess_return_vs_vti_{n}w"]
    mask = recomputed.notna() & reported.notna()
    assert np.allclose(recomputed[mask], reported[mask]), f"excess_return_vs_vti_{n}w disagrees with abs_return - vti_return"

# 7. No inf values anywhere in the feature columns.
inf_counts = np.isinf(feature_panel[FEATURE_COLUMNS].to_numpy(dtype=float)).sum()
assert inf_counts == 0, f"{inf_counts} inf values found in feature columns"

# 8. No unexpected NaNs once every window has had time to fill (first WARMUP_WEEKS weeks are
#    allowed to be NaN; every week after that must be fully populated).
sorted_dates = sorted(feature_panel["date"].unique())
post_warmup_dates = sorted_dates[WARMUP_WEEKS:]
post_warmup = feature_panel[feature_panel["date"].isin(post_warmup_dates)]
nan_counts = post_warmup[FEATURE_COLUMNS].isna().sum()
unexpected_nan = nan_counts[nan_counts > 0]
assert unexpected_nan.empty, f"unexpected NaNs after warmup: {unexpected_nan.to_dict()}"

print("all structural and cross-consistency checks passed")
print(f"  weeks: {feature_panel['date'].nunique()}, rows: {len(feature_panel)}, "
      f"warmup weeks excluded from NaN check: {WARMUP_WEEKS}")

all structural and cross-consistency checks passed
  weeks: 1141, rows: 12551, warmup weeks excluded from NaN check: 26


## Independent recomputation spot-check (20 random historical weeks)

The checks above operate on the panel `pandas` already produced with
`.rolling()`/`.shift()` — they can't catch a bug shared by both the
computation and the check (e.g. an off-by-one in a window that's wrong in
exactly the same way everywhere). This section recomputes every feature
for 20 randomly sampled `(date, symbol)` weeks using plain positional
`.iloc` slicing instead of `.rolling`/`.shift`, and asserts it matches the
panel exactly. Because each recomputation only ever reads prices up to
and including the sampled week's own index position, agreement here is
also a direct, per-row demonstration that **no target-period data reaches
any feature** for the sampled weeks — not just an assumption from reading
the code.

In [17]:
import random


def manual_recompute(date_label, symbol: str) -> dict:
    """Recompute one (date, symbol) row's features via plain .iloc slicing -- no .rolling/.shift."""
    i = sector_adj_close.index.get_loc(date_label)
    prices = sector_adj_close[symbol]
    vti_prices = vti_adj_close
    vols = sector_volume[symbol]

    result = {}
    for n in MOMENTUM_WINDOWS:
        result[f"abs_return_{n}w"] = prices.iloc[i] / prices.iloc[i - n] - 1

    vti_n_return = {n: vti_prices.iloc[i] / vti_prices.iloc[i - n] - 1 for n in MOMENTUM_WINDOWS}
    for n in MOMENTUM_WINDOWS:
        result[f"excess_return_vs_vti_{n}w"] = result[f"abs_return_{n}w"] - vti_n_return[n]

    for n in TREND_WINDOWS:
        window = prices.iloc[i - n + 1 : i + 1]
        result[f"trend_dist_from_ma_{n}w"] = prices.iloc[i] / window.mean() - 1

    for n in VOL_WINDOWS:
        log_returns = [np.log(prices.iloc[j] / prices.iloc[j - 1]) for j in range(i - n + 1, i + 1)]
        result[f"realized_vol_{n}w"] = float(np.std(log_returns, ddof=1))

    for n in DRAWDOWN_WINDOWS:
        window = prices.iloc[i - n + 1 : i + 1]
        result[f"drawdown_dist_from_high_{n}w"] = prices.iloc[i] / window.max() - 1

    result["volume_change_1w"] = vols.iloc[i] / vols.iloc[i - 1] - 1
    for n in (4, 12):
        prior_weeks = vols.iloc[i - n : i]  # t-1 .. t-n, current week excluded
        result[f"volume_vs_avg_{n}w"] = vols.iloc[i] / prior_weeks.mean() - 1

    sector_1w_return = {s: sector_adj_close[s].iloc[i] / sector_adj_close[s].iloc[i - 1] - 1 for s in SECTOR_ETFS}
    for n in RANK_WINDOWS:
        sector_n_return = {s: sector_adj_close[s].iloc[i] / sector_adj_close[s].iloc[i - n] - 1 for s in SECTOR_ETFS}
        ranks = pd.Series(sector_n_return).rank(ascending=False, method="average")
        result[f"cross_sectional_rank_{n}w"] = ranks[symbol]

    result["vti_return_1w"] = vti_n_return[1]
    result["vti_return_4w"] = vti_n_return[4]
    result["vti_return_12w"] = vti_n_return[12]
    vti_log_returns_4w = [np.log(vti_prices.iloc[j] / vti_prices.iloc[j - 1]) for j in range(i - 3, i + 1)]
    result["vti_volatility_4w"] = float(np.std(vti_log_returns_4w, ddof=1))

    result["sector_return_dispersion_1w"] = float(np.std(list(sector_1w_return.values()), ddof=1))
    result["sectors_outperforming_vti_1w"] = sum(1 for v in sector_1w_return.values() if v > vti_n_return[1])
    result["average_sector_return_1w"] = float(np.mean(list(sector_1w_return.values())))

    return result


random.seed(20260812)  # fixed seed -- reproducible sample, not cherry-picked
sample_positions = random.sample(range(WARMUP_WEEKS, len(sector_adj_close)), 20)
sample_dates = [sector_adj_close.index[i] for i in sample_positions]

mismatches = []
checked = 0
for date_label in sample_dates:
    panel_rows = feature_panel[feature_panel["date"] == date_label.strftime("%Y-%m-%d")]
    for symbol in SECTOR_ETFS:
        expected = manual_recompute(date_label, symbol)
        actual = panel_rows[panel_rows["symbol"] == symbol].iloc[0]
        for key, value in expected.items():
            checked += 1
            if not np.isclose(actual[key], value, rtol=1e-9, atol=1e-9):
                mismatches.append((date_label.date(), symbol, key, value, actual[key]))

assert not mismatches, f"{len(mismatches)} mismatches between manual recomputation and the panel: {mismatches[:5]}"
print(f"independent recomputation matched the panel on all {checked} (date, symbol, feature) values "
      f"across {len(sample_dates)} randomly sampled weeks x 11 sectors")

independent recomputation matched the panel on all 7920 (date, symbol, feature) values across 20 randomly sampled weeks x 11 sectors
